# CIP alerting system

Compares each actual CIP cycle in the station logs with the object's thresholds in `CIP Pre Rinse Timer details.xlsx` and lists every breach.

**Pipeline:** load logs → load thresholds and match objects → split cycles → compare with thresholds → alerts → summaries.

Every step is followed by validation checks. An **ERROR** stops the run; a **WARNING** is recorded and shown in the report at the end.

## 1. Settings
Paths, alert tolerances and the limits used by the validation checks. Edit them here.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path('datasets')
LOG_FILES = sorted(DATA_DIR.glob('CIP_Log*.xlsx'))
REFERENCE_FILE = DATA_DIR / 'CIP Pre Rinse Timer details.xlsx'

CYCLE_GAP_MINUTES = 30     # a longer gap between records starts a new cycle
DURATION_TOLERANCE = 0.10  # alert when actual duration > threshold + 10%
FLOW_TOLERANCE = 0.10      # alert when median flow < threshold - 10%

# Limits used by the validation checks
ACTIVITIES = ['Pre Rinse', 'Caustic', 'Mid Rinse', 'Acid', 'Final Rinse', 'Final Drain']
EXPECTED_RECIPES = {'Caustic', 'Caustic Acid', 'Acid', 'Final rinse'}
EXPECTED_STEPS = {'Fill To Tank', 'Fill Circuit', 'Front Mix Zone', 'Circulation',
                  'Empty Tank', 'Heating', 'Start Next Activity'}
VALID_RANGES = {  # physically plausible sensor ranges
    'Return Temp': (-5, 120),
    'Return Conducivity': (0, 250),
    'Supply Flow': (0, 60000),
    'Return Flow': (0, 60000),
}
SAMPLE_INTERVAL_SECONDS = 60
MIN_CYCLE_RECORDS = 5
MAX_CYCLE_HOURS = 4

if not LOG_FILES:
    raise FileNotFoundError(f'No CIP log files found in {DATA_DIR.resolve()}')
if not REFERENCE_FILE.exists():
    raise FileNotFoundError(f'Reference workbook not found: {REFERENCE_FILE.resolve()}')
print(f'{len(LOG_FILES)} log files, reference workbook: {REFERENCE_FILE.name}')

### Validation helpers
`check()` records each result in `validation_log`. Re-running a cell replaces its stage's earlier results.

In [ ]:
validation_log = []

def start_stage(stage):
    validation_log[:] = [row for row in validation_log if row['Stage'] != stage]

def preview(values, limit=5):
    values = [str(value) for value in values]
    more = f' (+{len(values) - limit} more)' if len(values) > limit else ''
    return ', '.join(values[:limit]) + more

def check(stage, name, passed, detail='', level='ERROR'):
    passed = bool(passed)
    validation_log.append({'Stage': stage, 'Check': name,
                           'Status': 'PASS' if passed else level,
                           'Detail': '' if passed else detail})
    if not passed:
        print(f'[{level}] {name}' + (f': {detail}' if detail else ''))
        if level == 'ERROR':
            raise AssertionError(f'{stage} check failed: {name}. {detail}')

def show_checks(stage):
    counts = pd.Series([row['Status'] for row in validation_log if row['Stage'] == stage]).value_counts()
    print(f'{stage}: ' + ', '.join(f'{count} {status}' for status, count in counts.items()))

start_stage('Config')
check('Config', 'Cycle gap is positive', CYCLE_GAP_MINUTES > 0, f'CYCLE_GAP_MINUTES = {CYCLE_GAP_MINUTES}')
for name, value in [('DURATION_TOLERANCE', DURATION_TOLERANCE), ('FLOW_TOLERANCE', FLOW_TOLERANCE)]:
    check('Config', f'{name} is between 0 and 1', 0 <= value < 1, f'{name} = {value}')
empty_files = [path.name for path in LOG_FILES if path.stat().st_size == 0]
check('Config', 'No empty log files', not empty_files, preview(empty_files))
show_checks('Config')

## 2. Load the CIP logs
Each workbook has a report header (line name, date range), column names on Excel row 14, a units row, then one record per minute. All files are combined into `logs`.

In [ ]:
REQUIRED = [
    'Date Time', 'Object', 'Recipe', 'Activity', 'Step',
    'Time In Seconds', 'Temp SP', 'Return Temp', 'Supply Flow',
    'Return Flow', 'Min Return Flow SP', 'Return Conducivity',
    'Caustic SP', 'Acid SP', 'Final Rinse Cond. SP', 'Flip'
]
HEADER_ROW = 13  # Excel row 14

load_stats = []  # one row per file, used by the checks below

def read_log(path):
    raw = pd.read_excel(path, sheet_name=0, header=None)
    header_text = ' '.join(raw.head(HEADER_ROW).astype(str).stack())
    line_match = re.search(r'Line\s*:\s*([^ ]+)', header_text, flags=re.I)
    line = line_match.group(1) if line_match else path.stem

    df = raw.iloc[HEADER_ROW + 1:].set_axis(raw.iloc[HEADER_ROW], axis=1)
    missing = [col for col in REQUIRED if col not in df.columns]
    if missing:
        raise ValueError(f'{path.name}: header row {HEADER_ROW + 1} is missing columns {missing}')
    df = df[REQUIRED].copy()
    rows_read = len(df)

    df['Date Time'] = pd.to_datetime(df['Date Time'], errors='coerce')
    df = df[df['Date Time'].notna()].copy()  # drops the units row and blank rows
    values_lost = 0
    for col in REQUIRED[5:]:
        had_value = df[col].notna()
        df[col] = pd.to_numeric(df[col], errors='coerce')
        values_lost += int((had_value & df[col].isna()).sum())

    df['Line'] = line
    df['Source File'] = path.name
    load_stats.append({'Source File': path.name, 'Line': line, 'Line From Header': bool(line_match),
                       'Rows Read': rows_read, 'Rows Without Timestamp': rows_read - len(df),
                       'Values Lost In Numeric Conversion': values_lost})
    return df

logs = pd.concat([read_log(path) for path in LOG_FILES], ignore_index=True)
logs = logs.sort_values(['Line', 'Object', 'Date Time']).reset_index(drop=True)
load_report = pd.DataFrame(load_stats)
print(f'{len(logs):,} records loaded')
load_report

**Checks:** file structure, missing or duplicate records, unknown names, sensor ranges, sampling rate and set points.

In [ ]:
start_stage('Load')
check('Load', 'Every log file has a line name in its header', load_report['Line From Header'].all(),
      preview(load_report.loc[~load_report['Line From Header'], 'Source File']))
check('Load', 'Each line comes from only one file', not load_report['Line'].duplicated().any(),
      preview(load_report.loc[load_report['Line'].duplicated(keep=False), 'Source File']))
check('Load', 'Only blank and unit rows lack a timestamp (at most 3 per file)',
      load_report['Rows Without Timestamp'].le(3).all(),
      preview(load_report.loc[load_report['Rows Without Timestamp'].gt(3), 'Source File']), level='WARNING')
check('Load', 'No values lost converting measurements to numbers',
      load_report['Values Lost In Numeric Conversion'].eq(0).all(),
      f"{load_report['Values Lost In Numeric Conversion'].sum()} values became NaN", level='WARNING')

for col in ['Object', 'Recipe', 'Activity', 'Step', 'Time In Seconds']:
    missing = logs[col].isna().sum()
    check('Load', f'{col} is filled on every record', missing == 0, f'{missing} records missing', level='WARNING')
duplicates = logs.duplicated(['Line', 'Object', 'Date Time']).sum()
check('Load', 'No duplicate records for the same object and timestamp', duplicates == 0,
      f'{duplicates} duplicates', level='WARNING')
for col, expected in [('Recipe', EXPECTED_RECIPES), ('Activity', set(ACTIVITIES)), ('Step', EXPECTED_STEPS)]:
    unknown = set(logs[col].dropna()) - expected
    check('Load', f'{col} values are all known', not unknown, preview(sorted(unknown)), level='WARNING')

check('Load', 'Elapsed time is never negative', logs['Time In Seconds'].ge(0).all(),
      f"{logs['Time In Seconds'].lt(0).sum()} negative values", level='WARNING')
for col, (low, high) in VALID_RANGES.items():
    outside = logs[col].notna() & ~logs[col].between(low, high)
    check('Load', f'{col} is within {low} to {high}', not outside.any(),
          f'{outside.sum():,} records ({outside.mean():.1%}) outside; '
          f'min {logs[col].min():.1f}, max {logs[col].max():.1f}', level='WARNING')

interval = logs.groupby(['Line', 'Object'])['Date Time'].diff().dt.total_seconds().median()
check('Load', f'Records are sampled about every {SAMPLE_INTERVAL_SECONDS} s',
      abs(interval - SAMPLE_INTERVAL_SECONDS) <= 5, f'median interval {interval:.0f} s', level='WARNING')

# Odd set-point records usually carry the previous object's value at a switch-over.
usual_set_point = logs.groupby('Object')['Min Return Flow SP'].transform(lambda s: s.mode().iloc[0])
odd_set_point = logs['Min Return Flow SP'].ne(usual_set_point)
check('Load', 'Min Return Flow SP is constant for each object', not odd_set_point.any(),
      f"{odd_set_point.sum()} records differ from their object's usual value", level='WARNING')

periods = logs.groupby('Line')['Date Time'].agg(['min', 'max'])
same_period = ((periods['min'].max() - periods['min'].min()) <= pd.Timedelta(days=1)
               and (periods['max'].max() - periods['max'].min()) <= pd.Timedelta(days=1))
check('Load', 'All lines cover the same period (within one day)', same_period,
      '; '.join(f'{line}: {row["min"]:%d %b} to {row["max"]:%d %b}' for line, row in periods.iterrows()),
      level='WARNING')
show_checks('Load')

## 3. Thresholds and object matching
Thresholds per object come from the reference workbook:

| Threshold | Source |
|---|---|
| Pre Rinse | Summary: `New Proposed Pre Rinse total time(Min)` |
| Caustic, Acid | Summary: `Caustic- Total Time`, `Acid Total Time` |
| Final Rinse | Process sheet: sum of the 4 final-rinse steps (seconds ÷ 60) |
| Supply / Return flow | Process sheet: `Suppy Flow (L/Hr)`, `Return Flow (L/Hr)` |

Log objects are matched to workbook objects by their equipment code, e.g. `C02C01-PMT TANK-1` → `M11W01_03_C02C01_PMT-1`.

In [ ]:
summary = pd.read_excel(REFERENCE_FILE, sheet_name='Summary')
process = pd.read_excel(REFERENCE_FILE, sheet_name='Process CIP Water Consumption')

FINAL_RINSE_SECONDS = ['Final Rinse Fill to Tank (sec)', 'Final Rinse Fill Circuit (Sec)',
                       'Final Rinse Front Mix Zone (Sec)', 'Final Rinse Circulation (Sec)']
DURATION_THRESHOLDS = {  # log activity -> threshold column
    'Pre Rinse': 'Pre Rinse Threshold (min)',
    'Caustic': 'Caustic Threshold (min)',
    'Acid': 'Acid Threshold (min)',
    'Final Rinse': 'Final Rinse Total Time (min)',
}
FLOWS = ['Supply', 'Return']
THRESHOLD_COLUMNS = [*DURATION_THRESHOLDS.values(), 'Supply Flow Threshold (L/hr)', 'Return Flow Threshold (L/hr)']

thresholds = summary.merge(process[['Object', 'Suppy Flow (L/Hr)', 'Return Flow (L/Hr)', *FINAL_RINSE_SECONDS]],
                           on='Object', how='left', validate='one_to_one')
thresholds['Final Rinse Total Time (min)'] = thresholds[FINAL_RINSE_SECONDS].sum(axis=1, min_count=4) / 60
thresholds = thresholds.rename(columns={
    'New Proposed Pre Rinse total time(Min)': 'Pre Rinse Threshold (min)',
    'Caustic- Total Time': 'Caustic Threshold (min)',
    'Acid Total Time': 'Acid Threshold (min)',
    'Suppy Flow (L/Hr)': 'Supply Flow Threshold (L/hr)',
    'Return Flow (L/Hr)': 'Return Flow Threshold (L/hr)',
})

def object_key(value):
    # Equipment code shared by log and workbook names, e.g. C02C01
    match = re.search(r'C\d{2}[A-Z]\d{2}', str(value).upper())
    return match.group(0) if match else str(value).strip().upper()

thresholds['Object Key'] = thresholds['Object'].map(object_key)
threshold_lookup = thresholds.set_index('Object')

def reference_object_for_log(log_object):
    candidates = thresholds.loc[thresholds['Object Key'].eq(object_key(log_object)), 'Object']
    if len(candidates) <= 1:
        return candidates.iloc[0] if len(candidates) else np.nan
    # Same code used twice (e.g. C10W02): pick the name sharing the most words with the log object
    log_text = str(log_object).upper()
    scores = candidates.map(lambda name: sum(token in log_text for token in re.findall(r'[A-Z0-9]+', name.upper())))
    return candidates[scores.idxmax()]

logs['Object Key'] = logs['Object'].map(object_key)
logs['Reference Object'] = logs['Object'].map({obj: reference_object_for_log(obj) for obj in logs['Object'].unique()})
print(f"{logs['Reference Object'].notna().mean():.0%} of records matched to a reference object")
thresholds[['Object', 'Object Key', *THRESHOLD_COLUMNS]]

**Checks:** workbook structure, missing values, whether Summary totals agree with the Process steps, equipment-code coverage, and whether the workbook flow equals the PLC's own set point.

In [ ]:
start_stage('Thresholds')
PHASE_STEP_COLUMNS = {
    'Pre Rinse': ['Pre Rinse Fill to Tank (Sec)', 'Pre Rinse Fill Circuit (Sec)',
                  'Pre Rinse Front Mix Zone (Sec)', 'PreRinse Circulation(sec)'],
    'Caustic': ['Caustic Fill to Tank (Sec)', 'Caustic Fill Circuit (Sec)',
                'Caustic Front Mix Zone (Sec)', 'Caustic Circulation (Sec)'],
    'Mid Rinse': ['Mid Rinse Fill to Tank (Sec)', 'Mid Rinse Fill Circuit (Sec)',
                  'Mid Rinse Front Mix Zone (Sec)', 'Mid rinse Circulation (sec)'],
    'Acid': ['Acid Circulation'],
}
missing_columns = [col for cols in PHASE_STEP_COLUMNS.values() for col in cols if col not in process.columns]
check('Thresholds', 'Process sheet has every step column', not missing_columns, preview(missing_columns))
check('Thresholds', 'Summary has one row per object', not summary['Object'].duplicated().any(),
      preview(summary.loc[summary['Object'].duplicated(), 'Object']))
not_in_process = sorted(set(summary['Object']) - set(process['Object']))
check('Thresholds', 'Every Summary object is in the Process sheet', not not_in_process, preview(not_in_process))

missing_values = threshold_lookup[THRESHOLD_COLUMNS].isna().stack()
missing_values = missing_values[missing_values]
check('Thresholds', 'No missing threshold values', missing_values.empty,
      preview(f'{obj}: {col}' for obj, col in missing_values.index), level='WARNING')
not_positive = thresholds[THRESHOLD_COLUMNS].le(0).any(axis=1)
check('Thresholds', 'All thresholds are positive', not not_positive.any(),
      preview(thresholds.loc[not_positive, 'Object']))

# Summary totals should equal the sum of the matching Process step times
step_minutes = {phase: process.set_index('Object')[cols].sum(axis=1, min_count=len(cols)).reindex(summary['Object']) / 60
                for phase, cols in PHASE_STEP_COLUMNS.items()}
summary_by_object = summary.set_index('Object')
for label, summary_col, expected in [
    ('caustic time equals the Process caustic steps', 'Caustic- Total Time', step_minutes['Caustic']),
    ('current pre-rinse time equals the Process pre-rinse steps', 'Current-Total Pre rinse time(Min)', step_minutes['Pre Rinse']),
]:
    differs = ~np.isclose(summary_by_object[summary_col], expected, atol=0.05)
    check('Thresholds', f'Summary {label}', not differs.any(), preview(summary_by_object.index[differs]), level='WARNING')
acid_with_mid_rinse = np.isclose(summary_by_object['Acid Total Time'], step_minutes['Acid'] + step_minutes['Mid Rinse'], atol=0.05)
check('Thresholds', 'Acid threshold covers only the Acid activity it is compared with',
      np.isclose(summary_by_object['Acid Total Time'], step_minutes['Acid'], atol=0.05).all(),
      f'{acid_with_mid_rinse.sum()} of {len(summary)} Acid Total Time values equal Mid Rinse + Acid Circulation',
      level='WARNING')

duplicate_keys = thresholds.loc[thresholds['Object Key'].duplicated(keep=False), 'Object']
check('Thresholds', 'Each equipment code appears once in the workbook', duplicate_keys.empty,
      preview(duplicate_keys), level='WARNING')
without_code = thresholds.loc[~thresholds['Object Key'].str.fullmatch(r'C\d{2}[A-Z]\d{2}'), 'Object']
check('Thresholds', 'Every workbook object has an equipment code', without_code.empty,
      preview(without_code), level='WARNING')
unmatched_objects = sorted(logs.loc[logs['Reference Object'].isna(), 'Object'].unique())
check('Thresholds', 'Every log object has a reference threshold', not unmatched_objects,
      f"{logs['Reference Object'].notna().mean():.0%} of records matched; unmatched: {preview(unmatched_objects, 10)}",
      level='WARNING')

# The PLC logs its own minimum return flow set point, which should equal the workbook value
plc_set_point = logs.dropna(subset=['Reference Object']).groupby('Reference Object')['Min Return Flow SP'].median()
workbook_set_point = threshold_lookup['Return Flow Threshold (L/hr)'].reindex(plc_set_point.index)
set_point_mismatch = plc_set_point.index[~np.isclose(plc_set_point, workbook_set_point)]
check('Thresholds', 'Workbook return flow equals the PLC Min Return Flow SP', set_point_mismatch.empty,
      preview(set_point_mismatch), level='WARNING')
show_checks('Thresholds')

## 4. Cycles
A cycle is a continuous run of records for one object; a gap of more than `CYCLE_GAP_MINUTES` starts the next one.

`Time In Seconds` counts up through the whole cycle, so an **activity's time = last counter − first counter** in that activity. Flows are the median over the whole cycle.

In [ ]:
gap = logs.groupby(['Line', 'Object'])['Date Time'].diff().gt(pd.Timedelta(minutes=CYCLE_GAP_MINUTES))
logs['Cycle Number'] = gap.groupby([logs['Line'], logs['Object']]).cumsum().astype(int)
cycle_keys = ['Line', 'Object', 'Reference Object', 'Object Key', 'Cycle Number']
by_cycle = logs.groupby(cycle_keys, dropna=False)

cycle_info = by_cycle.agg(**{
    'Cycle Start': ('Date Time', 'min'),
    'Cycle End': ('Date Time', 'max'),
    'Record Count': ('Date Time', 'size'),
    'Recipes': ('Recipe', lambda s: ', '.join(sorted(set(s.dropna().astype(str))))),
    'Supply Flow Actual (L/hr)': ('Supply Flow', 'median'),
    'Return Flow Actual (L/hr)': ('Return Flow', 'median'),
})

activity_seconds = logs.groupby(cycle_keys + ['Activity'], dropna=False)['Time In Seconds'].agg(['min', 'max'])
activity_minutes = ((activity_seconds['max'] - activity_seconds['min']) / 60).where(lambda m: m >= 0)
activity_minutes = activity_minutes.unstack('Activity').reindex(columns=ACTIVITIES)

cycle_results = cycle_info.join(activity_minutes).reset_index()
cycle_minutes = (cycle_results['Cycle End'] - cycle_results['Cycle Start']).dt.total_seconds() / 60
print(f'{len(cycle_results):,} cycles across {cycle_results["Object"].nunique()} objects')
cycle_results.head()

**Checks:** every record lands in exactly one cycle, cycles don't overlap, activity times add up, and each cycle is one complete CIP run.

In [ ]:
start_stage('Cycles')
check('Cycles', 'Every record is assigned to a cycle', logs['Cycle Number'].notna().all())
check('Cycles', 'One row per cycle', not cycle_results.duplicated(cycle_keys).any())
check('Cycles', 'Cycle record counts add up to the log', cycle_results['Record Count'].sum() == len(logs),
      f"{cycle_results['Record Count'].sum():,} vs {len(logs):,} records")
check('Cycles', 'Cycle start is never after cycle end', cycle_results['Cycle Start'].le(cycle_results['Cycle End']).all())

ordered_cycles = cycle_results.sort_values(['Line', 'Object', 'Cycle Start'])
gap_to_previous = (ordered_cycles['Cycle Start'] - ordered_cycles.groupby(['Line', 'Object'])['Cycle End'].shift()).dropna()
check('Cycles', f'Cycles of the same object are more than {CYCLE_GAP_MINUTES} min apart',
      gap_to_previous.gt(pd.Timedelta(minutes=CYCLE_GAP_MINUTES)).all(),
      f'{gap_to_previous.le(pd.Timedelta(minutes=CYCLE_GAP_MINUTES)).sum()} cycles overlap or are too close')

# The counter restarts at every new CIP run, so a drop means two runs share one cycle
counter_drop = by_cycle['Time In Seconds'].diff().lt(0)
merged_runs = logs.loc[counter_drop, cycle_keys].drop_duplicates()
check('Cycles', 'Each cycle holds a single CIP run (elapsed counter never restarts)', merged_runs.empty,
      f"{len(merged_runs)} cycles hold more than one run: {preview(merged_runs['Object'].unique())}", level='WARNING')
repeated_activity = by_cycle['Activity'].agg(lambda s: s.ne(s.shift()).sum() > s.nunique())
check('Cycles', 'Each activity occurs in one continuous block per cycle', not repeated_activity.any(),
      f'{repeated_activity.sum()} cycles repeat an activity', level='WARNING')

check('Cycles', 'Activity times are never negative', cycle_results[ACTIVITIES].fillna(0).ge(0).all().all())
over_cycle = cycle_results[ACTIVITIES].sum(axis=1) > cycle_minutes + 1
check('Cycles', 'Activity times add up to no more than the cycle duration', not over_cycle.any(),
      f'{over_cycle.sum()} cycles', level='WARNING')
sample_gaps = cycle_results['Record Count'] < 0.9 * (cycle_minutes * 60 / SAMPLE_INTERVAL_SECONDS + 1)
check('Cycles', 'No missing samples inside cycles', not sample_gaps.any(),
      f'{sample_gaps.sum()} cycles have under 90% of the expected records', level='WARNING')
short_cycles = cycle_results['Record Count'] < MIN_CYCLE_RECORDS
check('Cycles', f'No cycle has fewer than {MIN_CYCLE_RECORDS} records', not short_cycles.any(),
      f'{short_cycles.sum()} cycles', level='WARNING')
long_cycles = cycle_minutes > MAX_CYCLE_HOURS * 60
check('Cycles', f'No cycle is longer than {MAX_CYCLE_HOURS} h', not long_cycles.any(),
      f"{long_cycles.sum()} cycles: {preview(cycle_results.loc[long_cycles, 'Object'].unique())}", level='WARNING')

chemical_without_rinse = cycle_results[['Caustic', 'Acid']].notna().any(axis=1) & cycle_results['Final Rinse'].isna()
check('Cycles', 'Every cycle with caustic or acid has a final rinse', not chemical_without_rinse.any(),
      f"{chemical_without_rinse.sum()} cycles: {preview(cycle_results.loc[chemical_without_rinse, 'Object'].unique())}",
      level='WARNING')
no_flow = cycle_results['Supply Flow Actual (L/hr)'].le(0) | cycle_results['Return Flow Actual (L/hr)'].le(0)
check('Cycles', 'Median cycle flows are positive', not no_flow.any(),
      f'{no_flow.sum()} cycles have a median flow at or below zero (pump-off readings are included)', level='WARNING')
show_checks('Cycles')

## 5. Compare with thresholds
- **Duration breach:** activity time > threshold × (1 + `DURATION_TOLERANCE`)
- **Flow breach:** median flow < threshold × (1 − `FLOW_TOLERANCE`)

Status is **ALERT** (≥ 1 breach), **OK**, or **NO REFERENCE** (object not in the workbook, so not checked).

In [ ]:
for phase, threshold_col in DURATION_THRESHOLDS.items():
    cycle_results[f'{phase} Threshold (min)'] = cycle_results['Reference Object'].map(threshold_lookup[threshold_col])
    cycle_results[f'{phase} Breach'] = (cycle_results[phase]
                                        > cycle_results[f'{phase} Threshold (min)'] * (1 + DURATION_TOLERANCE))
for flow in FLOWS:
    threshold_col = f'{flow} Flow Threshold (L/hr)'
    cycle_results[threshold_col] = cycle_results['Reference Object'].map(threshold_lookup[threshold_col])
    cycle_results[f'{flow} Flow Breach'] = (cycle_results[f'{flow} Flow Actual (L/hr)']
                                            < cycle_results[threshold_col] * (1 - FLOW_TOLERANCE))

breach_columns = [col for col in cycle_results.columns if col.endswith('Breach')]
cycle_results['Alert Count'] = cycle_results[breach_columns].sum(axis=1)
cycle_results['Status'] = np.select(
    [cycle_results['Reference Object'].isna(), cycle_results['Alert Count'] > 0],
    ['NO REFERENCE', 'ALERT'], default='OK')
cycle_results['Status'].value_counts()

**Checks:** every breach flag is recalculated independently, and known threshold issues are measured (acid threshold, proposed pre-rinse timer, flow medians, merged cycles).

In [ ]:
start_stage('Alert logic')
has_reference = cycle_results['Reference Object'].notna()
check('Alert logic', 'Cycles without a reference raise no alerts', cycle_results.loc[~has_reference, 'Alert Count'].eq(0).all())
check('Alert logic', 'Status is ALERT exactly when a matched cycle has a breach',
      cycle_results.loc[has_reference, 'Status'].eq('ALERT').eq(cycle_results.loc[has_reference, 'Alert Count'].gt(0)).all())
for phase in DURATION_THRESHOLDS:
    recalculated = cycle_results[phase] > cycle_results[f'{phase} Threshold (min)'] * (1 + DURATION_TOLERANCE)
    check('Alert logic', f'{phase} breach flags match a recalculation', recalculated.eq(cycle_results[f'{phase} Breach']).all())
    no_threshold = has_reference & cycle_results[phase].notna() & cycle_results[f'{phase} Threshold (min)'].isna()
    check('Alert logic', f'Every matched cycle with {phase} has a {phase} threshold', not no_threshold.any(),
          f"{no_threshold.sum()} cycles: {preview(cycle_results.loc[no_threshold, 'Object'].unique())}", level='WARNING')
for flow in FLOWS:
    recalculated = (cycle_results[f'{flow} Flow Actual (L/hr)']
                    < cycle_results[f'{flow} Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE))
    check('Alert logic', f'{flow} flow breach flags match a recalculation', recalculated.eq(cycle_results[f'{flow} Flow Breach']).all())

# Acid threshold = Mid Rinse + Acid, but the actual compared with it is Acid alone
acid_with_mid_rinse = (cycle_results[['Mid Rinse', 'Acid']].sum(axis=1).where(cycle_results['Acid'].notna())
                       > cycle_results['Acid Threshold (min)'] * (1 + DURATION_TOLERANCE))
check('Alert logic', 'Acid alerts are the same when Mid Rinse is included in the actual',
      acid_with_mid_rinse.sum() == cycle_results['Acid Breach'].sum(),
      f"{cycle_results['Acid Breach'].sum()} acid alerts now, {acid_with_mid_rinse.sum()} with Mid Rinse included",
      level='WARNING')

# Pre-rinse threshold is the proposed timer, applied after the log period
timer_change = pd.to_datetime(pd.read_excel(REFERENCE_FILE, sheet_name='Water clear')['Date of Change'], errors='coerce').min()
current_pre_rinse = cycle_results['Reference Object'].map(summary.set_index('Object')['Current-Total Pre rinse time(Min)'])
alerts_vs_current = (cycle_results['Pre Rinse'] > current_pre_rinse * (1 + DURATION_TOLERANCE)).sum()
check('Alert logic', 'Proposed pre-rinse timers were in use during the log period',
      pd.isna(timer_change) or logs['Date Time'].min() >= timer_change,
      f"logs end {logs['Date Time'].max():%d %b %Y}, proposed timers applied from {timer_change:%d %b %Y}; "
      f"{cycle_results['Pre Rinse Breach'].sum()} pre-rinse alerts vs proposed timer, {alerts_vs_current} vs timer in use",
      level='WARNING')

# Flow breaches should still hold when only positive circulation readings are used
circulation = logs[logs['Step'].eq('Circulation')]
for flow in FLOWS:
    col = f'{flow} Flow'
    flagged = cycle_results[cycle_results[f'{flow} Flow Breach']].set_index(cycle_keys)
    circulation_median = circulation[circulation[col] > 0].groupby(cycle_keys)[col].median().reindex(flagged.index)
    holds = circulation_median < flagged[f'{flow} Flow Threshold (L/hr)'] * (1 - FLOW_TOLERANCE)
    check('Alert logic', f'{flow} flow breaches hold during circulation', holds.all(),
          f'{holds.sum()} of {len(flagged)} still breach using positive circulation readings only', level='WARNING')

alerts_from_merged = cycle_results.merge(merged_runs, on=cycle_keys)['Alert Count'].gt(0).sum()
check('Alert logic', 'No alerts come from cycles holding several CIP runs', alerts_from_merged == 0,
      f'{alerts_from_merged} alerting cycles hold more than one run', level='WARNING')
show_checks('Alert logic')

## 6. Alerts
One row per breach, with the actual value and the threshold it was compared with.

In [ ]:
ALERT_RULES = (  # (alert type, breach flag, actual, threshold, unit)
    [(f'{phase} duration', f'{phase} Breach', phase, f'{phase} Threshold (min)', 'minutes') for phase in DURATION_THRESHOLDS]
    + [(f'{flow} flow', f'{flow} Flow Breach', f'{flow} Flow Actual (L/hr)', f'{flow} Flow Threshold (L/hr)', 'L/hr')
       for flow in FLOWS])

alerts = pd.concat([
    cycle_results.loc[cycle_results[breach], cycle_keys + ['Cycle Start', 'Cycle End', actual, threshold]]
    .rename(columns={actual: 'Actual Value', threshold: 'Threshold'})
    .assign(**{'Alert Type': alert_type, 'Unit': unit})
    for alert_type, breach, actual, threshold, unit in ALERT_RULES
]).sort_index(kind='stable').reset_index(drop=True)
alerts = alerts[cycle_keys + ['Cycle Start', 'Cycle End', 'Alert Type', 'Actual Value', 'Threshold', 'Unit']]

print(f"{len(alerts):,} breaches in {cycle_results['Status'].eq('ALERT').sum():,} cycles")
alerts['Alert Type'].value_counts()

In [ ]:
start_stage('Alerts')
check('Alerts', 'One alert row per breach flag', len(alerts) == cycle_results['Alert Count'].sum(),
      f"{len(alerts)} rows vs {cycle_results['Alert Count'].sum()} breach flags")
check('Alerts', 'No duplicate alerts', not alerts.duplicated(cycle_keys + ['Alert Type']).any())
beyond = np.where(alerts['Unit'].eq('minutes'),
                  alerts['Actual Value'] > alerts['Threshold'] * (1 + DURATION_TOLERANCE),
                  alerts['Actual Value'] < alerts['Threshold'] * (1 - FLOW_TOLERANCE))
check('Alerts', 'Every alert value is beyond its threshold and tolerance', beyond.all(), f'{(~beyond).sum()} rows')
alert_cycles = alerts[cycle_keys].drop_duplicates().merge(cycle_results[cycle_keys + ['Status']], on=cycle_keys, how='left')
check('Alerts', 'Every alert belongs to a cycle in ALERT status', alert_cycles['Status'].eq('ALERT').all())
check('Alerts', 'Every ALERT cycle has at least one alert row',
      len(alert_cycles) == cycle_results['Status'].eq('ALERT').sum())
show_checks('Alerts')
alerts.head(20)

## 7. Summaries
Cycles per object by status, and cycles and alerts per day.

In [ ]:
cycle_counts = pd.crosstab([cycle_results['Line'], cycle_results['Object']], cycle_results['Status'])
cycle_counts.insert(0, 'Cycles', cycle_counts.sum(axis=1))
display(cycle_counts)

alerts_per_day = (cycle_results.assign(Date=cycle_results['Cycle Start'].dt.date)
                  .groupby('Date')
                  .agg(**{'Cycles': ('Status', 'size'),
                          'Cycles with Alerts': ('Status', lambda s: s.eq('ALERT').sum()),
                          'Alert Count': ('Alert Count', 'sum')})
                  .reset_index())

start_stage('Summaries')
check('Summaries', 'Cycle counts add up to all cycles', cycle_counts['Cycles'].sum() == len(cycle_results))
check('Summaries', 'Daily cycles add up to all cycles', alerts_per_day['Cycles'].sum() == len(cycle_results))
check('Summaries', 'Daily alerts add up to all alerts', alerts_per_day['Alert Count'].sum() == len(alerts))
check('Summaries', 'Daily cycles with alerts add up to ALERT cycles',
      alerts_per_day['Cycles with Alerts'].sum() == cycle_results['Status'].eq('ALERT').sum())
show_checks('Summaries')
alerts_per_day

## 8. Validation report
Every warning from the run. Each one marks a result that is less reliable until the issue is fixed or confirmed with the CIP owner.

In [ ]:
validation_report = pd.DataFrame(validation_log)
display(validation_report.groupby(['Stage', 'Status'], sort=False).size().unstack(fill_value=0))
validation_report[validation_report['Status'].ne('PASS')].reset_index(drop=True)

## Open points
1. **Acid threshold** includes Mid Rinse; compare it with Mid Rinse + Acid, or use Acid Circulation alone.
2. **Cycles** should also split when `Time In Seconds` restarts, not only on a 30-minute gap.
3. **Flow** should use positive readings from circulation steps, not the whole-cycle median.
4. **Object mapping** for RMT tanks, Milk Reception, Raw Milk Tanker and Milk Past-2 (the workbook says `C10W02`).
5. **Pre-rinse threshold** for July data should be the timer in use then, not the proposed one.
6. **Tolerances** (10%) are working defaults, not agreed plant rules.